# Training data preparation

This notebook runs the reproducible pipeline that creates the four training data files: train.csv, train_anomaly_cleaned.csv, train_final.csv, and train_final_physics.csv.

The former exploratory and analysis cells are preserved in archive/train_full_legacy.ipynb. Analysis output tables are stored in analysis/.


In [1]:
import pandas as pd

wave = pd.read_csv("train_wave.csv")
atmos = pd.read_csv("train_atmos.csv")

wave["time"] = pd.to_datetime(wave["time"])
atmos["time"] = pd.to_datetime(atmos["time"])

print("=== WAVE ===")
print("rows:", len(wave))
print(
    "duplicate station-time rows:",
    wave.duplicated(
        ["station", "time"],
        keep=False
    ).sum()
)

print(
    "duplicate keys:",
    wave.loc[
        wave.duplicated(
            ["station", "time"],
            keep=False
        ),
        ["station", "time"]
    ]
    .drop_duplicates()
    .shape[0]
)


print("\n=== ATMOS ===")
print("rows:", len(atmos))
print(
    "duplicate station-time rows:",
    atmos.duplicated(
        ["station", "time"],
        keep=False
    ).sum()
)

print(
    "duplicate keys:",
    atmos.loc[
        atmos.duplicated(
            ["station", "time"],
            keep=False
        ),
        ["station", "time"]
    ]
    .drop_duplicates()
    .shape[0]
)


# 문제 시각 직접 확인
target_station = "S-ORS"
target_time = pd.Timestamp(
    "2025-06-30 13:30:00+09:00"
)

print("\n=== WAVE TARGET ===")
display(
    wave[
        (wave["station"] == target_station)
        &
        (wave["time"] == target_time)
    ]
)

print("\n=== ATMOS TARGET ===")
display(
    atmos[
        (atmos["station"] == target_station)
        &
        (atmos["time"] == target_time)
    ]
)

=== WAVE ===
rows: 118152
duplicate station-time rows: 0
duplicate keys: 0

=== ATMOS ===
rows: 130896
duplicate station-time rows: 0
duplicate keys: 0

=== WAVE TARGET ===


,station,time,hs,tp,hmax,wvdir



=== ATMOS TARGET ===


,station,time,wspd,gust,wdir,airt,relh,caph
130833,S-ORS,2025-06-30 13:30:00+09:00,6.03,6.468,167.54,23.77,NaN,1006.296


In [3]:
import pandas as pd

# ============================================================
# 1. LOAD
# ============================================================

wave = pd.read_csv("train_wave.csv")
atmos = pd.read_csv("train_atmos.csv")

wave["time"] = pd.to_datetime(wave["time"])
atmos["time"] = pd.to_datetime(atmos["time"])


# ============================================================
# 2. MERGE
# 같은 station + time 기준으로 합치기
# 한쪽에 없으면 NaN 유지
# ============================================================

df = pd.merge(
    wave,
    atmos,
    on=["station", "time"],
    how="outer",
    validate="one_to_one"
)


# ============================================================
# 3. COLUMN ORDER
# ============================================================

cols = [
    "station",
    "time",
    "hs",
    "tp",
    "hmax",
    "wvdir",
    "wspd",
    "gust",
    "wdir",
    "airt",
    "relh",
    "caph",
]

df = df[cols]


# ============================================================
# 4. SORT
# ============================================================

df = (
    df
    .sort_values(["station", "time"])
    .reset_index(drop=True)
)


# ============================================================
# 5. SANITY CHECK
# ============================================================

assert not df.duplicated(
    ["station", "time"]
).any(), "station + time 중복 발생"


print("shape:", df.shape)
print("\nNaN count:")
print(df.isna().sum())

display(df.head())


# ============================================================
# 6. SAVE
# ============================================================

df.to_csv(
    "train.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved: train.csv")

shape: (183600, 12)

NaN count:
station        0
time           0
hs         75670
tp         77487
hmax       75478
wvdir      74754
wspd       55364
gust       55348
wdir       55311
airt       55468
relh       69920
caph       55024
dtype: int64


,station,time,hs,tp,hmax,wvdir,wspd,gust,wdir,airt,relh,caph
0,G-ORS,2024-01-01 00:00:00+09:00,2.29,7.01,3.72,334.22,8.180,9.65,350.2,7.458,60.48,1023.736
1,G-ORS,2024-01-01 00:10:00+09:00,NaN,NaN,NaN,NaN,9.200,10.87,323.9,7.505,62.19,1023.653
2,G-ORS,2024-01-01 00:20:00+09:00,2.27,7.10,3.69,332.58,7.977,10.05,322.7,7.561,59.82,1023.708
3,G-ORS,2024-01-01 00:30:00+09:00,NaN,NaN,NaN,NaN,9.310,10.57,326.9,7.764,58.55,1023.673
4,G-ORS,2024-01-01 00:40:00+09:00,2.33,8.19,3.79,327.86,6.713,7.72,323.0,7.541,57.95,1023.717



Saved: train.csv


In [3]:
# ============================================================
# 7. REGULAR 10-MINUTE GRID + OBSERVATION MASKS (NO IMPUTATION)
# ============================================================
# Keep the raw union in train.csv. This file only inserts missing
# timestamps; it never fabricates a measurement.
import pandas as pd


INPUT_PATH = "train.csv"
OUTPUT_PATH = "train_v2.csv"
KEYS = ["station", "time"]
WAVE_COLUMNS = ["hs", "tp", "hmax", "wvdir"]
ATMOS_COLUMNS = ["wspd", "gust", "wdir", "airt", "relh", "caph"]
VALUE_COLUMNS = WAVE_COLUMNS + ATMOS_COLUMNS

raw = pd.read_csv(INPUT_PATH, parse_dates=["time"])
assert not raw.duplicated(KEYS).any(), "Duplicate station-time rows found"

def regularize_station(group):
    g = group.sort_values("time").copy()
    station = g["station"].iloc[0]

    # These flags describe original observations, before grid expansion.
    g["wave_available"] = g[WAVE_COLUMNS].notna().any(axis=1).astype("int8")
    g["atmos_available"] = g[ATMOS_COLUMNS].notna().any(axis=1).astype("int8")
    for col in VALUE_COLUMNS:
        g[f"{col}_observed"] = g[col].notna().astype("int8")

    grid = pd.date_range(g["time"].min(), g["time"].max(), freq="10min")
    g = g.set_index("time").reindex(grid)
    g.index.name = "time"
    g["grid_inserted"] = g["station"].isna().astype("int8")
    g["station"] = station

    flag_columns = ["wave_available", "atmos_available"] + [
        f"{col}_observed" for col in VALUE_COLUMNS
    ]
    g[flag_columns] = g[flag_columns].fillna(0).astype("int8")
    return g.reset_index()

regularized = pd.concat(
    [regularize_station(group) for _, group in raw.groupby("station", sort=True)],
    ignore_index=True,
)
regularized = regularized.sort_values(KEYS).reset_index(drop=True)

assert not regularized.duplicated(KEYS).any(), "Duplicate station-time rows created"
for _, group in regularized.groupby("station"):
    assert group["time"].diff().dropna().eq(pd.Timedelta(minutes=10)).all()

regularized.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

summary = regularized.groupby("station", as_index=False).agg(
    rows=("time", "size"),
    grid_inserted=("grid_inserted", "sum"),
    wave_available=("wave_available", "sum"),
    atmos_available=("atmos_available", "sum"),
)
print(f"Saved: {OUTPUT_PATH} | shape: {regularized.shape}")
display(summary)


Saved: train_v2.csv | shape: (236304, 25)


,station,rows,grid_inserted,wave_available,atmos_available
0,G-ORS,78768,0,38479,76848
1,I-ORS,78768,26352,31065,26064
2,S-ORS,78768,26352,39302,26063


In [ ]:
# ============================================================
# 8. SIMPLE REGULAR 10-MINUTE TRAIN V2 (NO QA MASK COLUMNS)
# ============================================================
# train.csv remains the raw source. train_v2.csv only completes the
# station-wise time index and intentionally keeps missing values as NaN.

import pandas as pd

BASE_COLUMNS = [
    "station", "time", "hs", "tp", "hmax", "wvdir",
    "wspd", "gust", "wdir", "airt", "relh", "caph",
]
raw = pd.read_csv("train.csv", parse_dates=["time"])[BASE_COLUMNS]

def add_regular_time_rows(group):
    g = group.sort_values("time").copy()
    station = g["station"].iloc[0]
    grid = pd.date_range(g["time"].min(), g["time"].max(), freq="10min")
    g = g.set_index("time").reindex(grid)
    g.index.name = "time"
    g["station"] = station
    return g.reset_index()[BASE_COLUMNS]

train_v2 = pd.concat(
    [add_regular_time_rows(group) for _, group in raw.groupby("station", sort=True)],
    ignore_index=True,
).sort_values(["station", "time"]).reset_index(drop=True)

assert train_v2.columns.tolist() == BASE_COLUMNS
assert not train_v2.duplicated(["station", "time"]).any()
train_v2.to_csv("train_v2.csv", index=False, encoding="utf-8")
print(f"Saved train_v2.csv: {train_v2.shape}")
print(train_v2.groupby("station").size())
